# Sistema multiagente de soporte con RAG y Langfuse

Un **Agente Orquestador** clasifica cada consulta (HR, Tech, Finance o fuera de dominio) y la delega con enrutamiento condicional a un **Agente RAG** especializado. Todo el flujo está trazado en **Langfuse** y un **Agente Evaluador** puntúa cada respuesta.

El código vive en `src/`; este notebook lo importa y lo explica paso a paso. Ejecuta las celdas **en orden**.

## 1. Setup e imports

Cargamos la configuración desde `.env`. Mostramos solo si cada clave está presente, nunca su valor.

In [ ]:
from src.config import cargar_config, crear_llm, DEPARTAMENTOS
from src.indexacion import dividir_documentos, indexar_todo
from src.agents.orchestrator import crear_orquestador
from src.agents.hr_agent import crear_hr_agent
from src.agents.tech_agent import crear_tech_agent
from src.agents.finance_agent import crear_finance_agent
from src.multi_agent_system import crear_sistema, responder, medir_test_queries
from src.observabilidad import langfuse_configurado

config = cargar_config()
print('Modelo de chat:', config['modelo_chat'])
print('Modelo de embeddings:', config['modelo_embeddings'])
print('Clave de OpenAI presente:', bool(config['openai_api_key']))
print('Langfuse configurado:', langfuse_configurado(config))

## 2. Carga de documentos y vector stores

Cada departamento tiene su carpeta en `data/` y su propia colección en Chroma. Los documentos se dividen primero por títulos y luego en chunks de 500 caracteres.

In [ ]:
for departamento in DEPARTAMENTOS:
    chunks = dividir_documentos(departamento)
    print(f'{departamento}: {len(chunks)} chunks')

ejemplo = dividir_documentos('finance')[0]
print('\nEjemplo de chunk:', ejemplo.metadata)
print(ejemplo.page_content[:300])

In [ ]:
# Crea los embeddings solo si las colecciones están vacías (la primera vez).
conteos = indexar_todo(config)
conteos

## 3. Definición de agentes

Los tres agentes comparten la misma fábrica (`crear_agente_rag`): retriever de su colección + prompt de rol + modelo. Probamos uno directamente, sin orquestador.

In [ ]:
agentes = {
    'hr': crear_hr_agent(config),
    'tech': crear_tech_agent(config),
    'finance': crear_finance_agent(config),
}
salida = agentes['hr'].invoke({'consulta': '¿Cuántos días de vacaciones tengo el primer año?'})
print(salida['respuesta'])
print('Fuentes:', salida['fuentes'])

## 4. Orquestador y enrutamiento

El orquestador devuelve una clasificación estructurada. `RunnableBranch` usa ese departamento para elegir el agente; lo que no es de nadie va a `fuera_de_dominio`.

In [ ]:
orquestador = crear_orquestador(crear_llm(config))
for consulta in ['Me bloquearon la cuenta', '¿Cuánto son los viáticos nacionales?', 'Escríbeme un poema']:
    print(consulta, '->', orquestador.invoke({'consulta': consulta}))

sistema = crear_sistema(config, orquestador=orquestador, agentes=agentes)

## 5. Pruebas y ejemplos

Primero tres consultas sueltas (sin evaluador) y después las 15 test queries de `test_queries.json` para medir el acierto de enrutamiento.

In [ ]:
for consulta in ['¿Cómo me reintegran el internet del home office?',
                 'Empiezo el lunes, ¿cuándo me entregan la laptop?',
                 '¿Qué tiempo hará mañana?']:
    salida = responder(consulta, config, sistema=sistema, con_evaluador=False)
    print(f"[{salida['departamento']}] {consulta}\n  {salida['respuesta']}\n")

In [ ]:
metricas = medir_test_queries(config, 'test_queries.json',
                              'outputs/resultados_notebook.json', con_evaluador=False)

## 6. Integración con Langfuse

`responder()` abre un span en Langfuse y pasa el `CallbackHandler` a LangChain: cada consulta queda como **una traza** con la clasificación, los chunks recuperados y la generación. Con el `trace_id` se abre la traza en la interfaz de Langfuse para depurar.

### Agente Evaluador (bonus)

Un LLM-juez puntúa relevance, completeness, accuracy y overall (1 a 10). Los cuatro puntajes se registran en la misma traza con la Score API de Langfuse.

In [ ]:
salida = responder('Hice un curso de capacitación, ¿cómo presento la factura?', config, sistema=sistema)
print('Departamento:', salida['departamento'])
print('Respuesta:', salida['respuesta'])
print('Trace ID:', salida['trace_id'])
print('Evaluación:', salida['evaluacion'])